# Airline Ticket Price Prediction — Final Colab Notebook

**Beginner-friendly practical notebook** for cleaning, visualization, statistics, and ticket-price prediction.

**Dataset:** Indian Flight Price Prediction dataset: https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction

Upload the extracted CSV when prompted. The dataset contains historical prices, not live fares. The prediction model now uses **source city** and **destination city** along with flight duration, days left, and number of stops.

## Import libraries

In [ ]:
import pandas as pd
import numpy as np

## Load the airline dataset

In [ ]:
from google.colab import files

uploaded = files.upload()  # Choose the airline CSV file
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Rows and columns:", df.shape)
print(df.head())

##  — Data cleaning and descriptive statistics

In [ ]:
# See missing values and duplicate rows
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows where ticket price is missing
df = df.dropna(subset=["price"])

# Basic statistics for ticket price
print("Mean:", df["price"].mean())
print("Median:", df["price"].median())
print("Mode:", df["price"].mode()[0])
print("Variance:", df["price"].var())
print("Standard deviation:", df["price"].std())
print("Quartiles:")
print(df["price"].quantile([0.25, 0.50, 0.75]))

# Summary statistics
print(df.describe())

##  — Data visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Histogram
plt.hist(df["price"], bins=30)
plt.title("Distribution of Ticket Prices")
plt.xlabel("Ticket price")
plt.ylabel("Number of flights")
plt.show()

# 2. Box plot
plt.boxplot(df["price"])
plt.title("Ticket Price Outliers")
plt.ylabel("Ticket price")
plt.show()

# 3. Bar chart: average price by airline
df.groupby("airline")["price"].mean().plot(kind="bar")
plt.title("Average Price by Airline")
plt.ylabel("Average price")
plt.show()

# 4. Pie chart: number of flights by airline
df["airline"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Flights by Airline")
plt.ylabel("")
plt.show()

# 5. Scatter plot: days left vs price
plt.scatter(df["days_left"], df["price"], alpha=0.3)
plt.title("Days Left vs Ticket Price")
plt.xlabel("Days left before departure")
plt.ylabel("Ticket price")
plt.show()

# 6. Line chart: average price by days left
df.groupby("days_left")["price"].mean().sort_index().plot(kind="line")
plt.title("Average Ticket Price by Days Left")
plt.xlabel("Days left")
plt.ylabel("Average price")
plt.show()

## — Probability and sampling

In [ ]:
import numpy as np

# Probability that a randomly selected ticket costs more than the mean
mean_price = df["price"].mean()
probability = (df["price"] > mean_price).mean()
print("Probability price is above the mean:", probability)

# Random sample of 10 flights
random_sample = df.sample(n=10, random_state=1)
print(random_sample[["airline", "price"]])

# Sample ticket prices and view their distribution
plt.hist(random_sample["price"], bins=5)
plt.title("Prices in a Random Sample")
plt.xlabel("Ticket price")
plt.show()

##  — Central Limit Theorem and point estimation

In [ ]:
# Take 100 samples and calculate each sample's mean price
sample_means = []

for i in range(100):
    sample = df["price"].sample(n=30, replace=True, random_state=i)
    sample_means.append(sample.mean())

print("Population mean estimate:", df["price"].mean())
print("Mean of sample means:", np.mean(sample_means))

plt.hist(sample_means, bins=15)
plt.title("Central Limit Theorem: Sample Means")
plt.xlabel("Sample mean price")
plt.ylabel("Frequency")
plt.show()

##  — T-test
We compare ticket prices for nonstop and connecting flights. The code checks the dataset's `stops` values and assumes `zero` means nonstop, as in the common Kaggle dataset.

In [ ]:
from scipy.stats import ttest_ind

print(df["stops"].value_counts())

# Separate nonstop and connecting flight prices
nonstop = df[df["stops"].astype(str).str.lower() == "zero"]["price"]
connecting = df[df["stops"].astype(str).str.lower() != "zero"]["price"]

# Welch's independent t-test
t_value, p_value = ttest_ind(nonstop, connecting, equal_var=False)

print("Average nonstop price:", nonstop.mean())
print("Average connecting price:", connecting.mean())
print("T statistic:", t_value)
print("P value:", p_value)

if p_value < 0.05:
    print("There is a statistically significant difference in average prices.")
else:
    print("There is not enough evidence of a difference in average prices.")

##   — Model Comparison with Source and Destination City

This section predicts ticket price using nine inputs: flight duration, days left before departure, number of stops, airline, class (Economy or Business), departure time, arrival time, source city, and destination city. Names such as airlines and cities are text, so we use **One-Hot Encoding** to convert them into model-friendly numeric features. A pipeline keeps the preprocessing and model together.

We train **three different models** on the same data and compare them:
- **Decision Tree** — a single tree of if/else rules.
- **Random Forest** — many decision trees, predictions averaged.
- **Gradient Boosting** — trees built one after another, each fixing the errors of the previous ones.

The best model (highest R² score on the test data) is used for the Streamlit app.

### Prepare the data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Convert duration into minutes
# (works for hours such as 2.17 and for text such as '2h 50m')
def convert_duration(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    try:
        return float(text) * 60          # plain number = hours
    except ValueError:
        pass
    hours = 0
    minutes = 0
    for part in text.split():
        if "h" in part:
            try:
                hours = int(part.replace("h", ""))
            except ValueError:
                pass
        elif "m" in part:
            try:
                minutes = int(part.replace("m", ""))
            except ValueError:
                pass
    return hours * 60 + minutes

# Convert duration and stops into numbers
df["duration_minutes"] = df["duration"].apply(convert_duration)
df["stops_number"] = df["stops"].astype(str).str.lower().map({
    "zero": 0, "one": 1, "two_or_more": 2,
    "non-stop": 0, "nonstop": 0, "0": 0, "1": 1, "2": 2
})

# Input columns used by the model
numeric_features = ["duration_minutes", "days_left", "stops_number"]
text_features = ["airline", "class", "departure_time", "arrival_time",
                 "source_city", "destination_city"]

# Check that the dataset contains the columns needed by the model
missing_columns = [col for col in text_features if col not in df.columns]
if missing_columns:
    raise ValueError(
        "Missing required column(s): " + ", ".join(missing_columns) +
        ". Please upload a flight dataset that contains these columns."
    )

# Prepare a clean dataset for the model
model_data = df[numeric_features + text_features + ["price", "stops"]].copy()
model_data["days_left"] = pd.to_numeric(model_data["days_left"], errors="coerce")
model_data["price"] = pd.to_numeric(model_data["price"], errors="coerce")
for col in text_features:
    model_data[col] = model_data[col].astype("string").str.strip()
model_data = model_data.dropna(subset=numeric_features + text_features + ["price"])

# X = input features; y = the ticket price we want to predict
X = model_data[numeric_features + text_features]
y = model_data["price"]

# Fill missing numeric values and convert text columns with One-Hot Encoding
preprocessor = ColumnTransformer(transformers=[
    ("numbers", SimpleImputer(strategy="median"), numeric_features),
    ("text", Pipeline(steps=[
        ("fill_missing_text", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ]), text_features)
])

# Split data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Rows used for model:", len(model_data))
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

### Train the three models

In [ ]:
# Three models to compare
models = {
    "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=12, min_samples_leaf=5,
                                           n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42),
}

trained_models = {}
all_predictions = {}
result_rows = []

for name, regressor in models.items():
    # Same preprocessing for every model, then the model itself
    pipe = Pipeline(steps=[
        ("preprocessing", clone(preprocessor)),
        ("regression", regressor)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    trained_models[name] = pipe
    all_predictions[name] = pred
    result_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 score": r2_score(y_test, pred),
    })
    print(name, "trained")

### Compare the models

In [ ]:
results_df = pd.DataFrame(result_rows).sort_values("R2 score", ascending=False).reset_index(drop=True)
results_df = results_df.round(3)
print(results_df)

best_model_name = results_df.loc[0, "Model"]
print("\nBest model (highest R2 score):", best_model_name)

In [ ]:
# Bar charts: lower MAE / RMSE and higher R2 are better
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["MAE", "RMSE", "R2 score"]):
    ax.bar(results_df["Model"], results_df[metric])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

### Best model: actual vs predicted prices

In [ ]:
# Use the best model for the rest of the notebook and the app
model = trained_models[best_model_name]
predictions = all_predictions[best_model_name]

results = pd.DataFrame({"Actual price": y_test.values, "Predicted price": predictions})
print(results.head())

### Check: does the number of stops change the price?
Tree-based models only change their answer when a split in the tree uses that feature. This check shows the data and every model's prediction for the **same flight** with 0, 1 and 2 stops.

In [ ]:
# Average price and number of flights for each number of stops
print(model_data.groupby("stops_number")["price"].agg(["mean", "count"]).round(0))

# Same flight, only the number of stops changes
example = X_test.iloc[[0]].copy()
rows = {}
for name, pipe in trained_models.items():
    rows[name] = [pipe.predict(example.assign(stops_number=s))[0] for s in [0, 1, 2]]

print("\nPredicted price for the same flight with 0, 1 and 2 stops:")
print(pd.DataFrame(rows, index=["0 stops", "1 stop", "2+ stops"]).round(0))

In [ ]:
import joblib

# Save all trained models and the details the app needs
all_cities = sorted(set(model_data["source_city"].astype(str)) | set(model_data["destination_city"].astype(str)))

joblib.dump({
    "models": trained_models,
    "results": results_df.to_dict("records"),
    "best": best_model_name,
    "cities": all_cities,
    "options": {col: sorted(model_data[col].astype(str).unique().tolist())
                for col in ["airline", "class", "departure_time", "arrival_time"]},
}, "airline_model.pkl", compress=3)

print("Models saved as airline_model.pkl")
print("Cities saved:", len(all_cities))
print(all_cities)

##  — Streamlit prediction interface

The app loads the saved models, shows the model comparison, and predicts a ticket price with the model the user chooses. It is written to `app.py` in small pieces (the first cell creates the file, the next cells add to it).

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy joblib

### Streamlit app — part 1: load the model

In [ ]:
%%writefile app.py

import os
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Airline Ticket Price Prediction", page_icon="✈️")
st.title("✈️ Airline Ticket Price Prediction")
st.write("Compares Decision Tree, Random Forest and Gradient Boosting models trained on the Indian Flight Price dataset (historical prices, not live fares).")

# Load the models saved from the notebook
model_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "airline_model.pkl")

# The file's modified time is part of the cache key, so a newly saved model is loaded automatically
@st.cache_resource
def load_model(last_modified):
    return joblib.load(model_path)

saved = load_model(os.path.getmtime(model_path))
models = saved["models"]
cities = saved["cities"]
options = saved["options"]
results = pd.DataFrame(saved["results"])
best_model_name = saved["best"]

### Streamlit app — part 2: model comparison

In [ ]:
%%writefile -a app.py

st.subheader("Model comparison")
st.dataframe(results, hide_index=True)
st.bar_chart(results.set_index("Model")["R2 score"])
st.success(f"Best model: {best_model_name} (highest R2 score)")

### Streamlit app — part 3: user inputs

In [ ]:
%%writefile -a app.py

st.subheader("Enter flight details")
st.caption(f"Model trained on {len(cities)} cities and {len(options['airline'])} airlines.")

model_name = st.selectbox("Choose model", list(models.keys()),
                          index=list(models.keys()).index(best_model_name))

col1, col2 = st.columns(2)
airline = col1.selectbox("Airline", options["airline"])
travel_class = col2.selectbox("Class", options["class"])

col1, col2 = st.columns(2)
source_city = col1.selectbox("Source city", cities)
destination_city = col2.selectbox("Destination city", cities, index=1)

col1, col2 = st.columns(2)
departure_time = col1.selectbox("Departure time", options["departure_time"])
arrival_time = col2.selectbox("Arrival time", options["arrival_time"])

stops = st.selectbox("Number of stops", ["zero", "one", "two_or_more"])

hours = st.number_input("Flight duration (hours)", 0.5, 50.0, 2.0, step=0.25)

days_left = st.slider("Days left before departure", 1, 49, 15)

### Streamlit app — part 4: predict the price

In [ ]:
%%writefile -a app.py

stops_number = {"zero": 0, "one": 1, "two_or_more": 2}[stops]
duration_minutes = hours * 60

if st.button("Predict Price"):
    if source_city == destination_city:
        st.error("Source city and destination city must be different.")
    else:
        new_flight = pd.DataFrame([{
            "duration_minutes": duration_minutes,
            "days_left": days_left,
            "stops_number": stops_number,
            "airline": airline,
            "class": travel_class,
            "departure_time": departure_time,
            "arrival_time": arrival_time,
            "source_city": source_city,
            "destination_city": destination_city,
        }])
        price = models[model_name].predict(new_flight)[0]
        mae = results.loc[results["Model"] == model_name, "MAE"].iloc[0]
        st.success(f"Estimated ticket price ({model_name}): ₹ {price:,.0f}")
        st.caption(f"On test data this model's average error is about ₹ {mae:,.0f}.")

        # Show the price for every number of stops (all other inputs unchanged)
        compare = {}
        for label, number in [("zero", 0), ("one", 1), ("two_or_more", 2)]:
            row = new_flight.assign(stops_number=number)
            compare[label] = [f"₹ {models[model_name].predict(row)[0]:,.0f}"]
        st.write("Price for each number of stops (same other details):")
        st.table(pd.DataFrame(compare, index=[model_name]))
        if len(set(v[0] for v in compare.values())) == 1:
            st.info("This model gives the same price for every number of stops for these inputs, "
                    "so it is not using stops here. Try another model.")

### Run the Streamlit app in Colab
Run these cells in order. The app opens inside the last cell.

In [ ]:
import subprocess

# Stop any old Streamlit app so the new one starts cleanly
subprocess.run(["pkill", "-f", "streamlit"], check=False)
print("Old Streamlit app stopped (if it was running)")

In [ ]:
!streamlit run app.py \
  --server.port 8502 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --browser.gatherUsageStats false \
  > streamlit8502.log 2>&1 &

In [ ]:
import time, urllib.request

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8502/_stcore/health", timeout=2)
        print("Streamlit is running on port 8502")
        break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit did not start. Log:")
    print(open("streamlit8502.log").read()[-3000:])

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8502, height=900)

### Run the app on your own computer (optional)
Download `app.py` and `airline_model.pkl`, keep them in one folder, then run:

```bash
pip install streamlit scikit-learn pandas numpy joblib
python -m streamlit run app.py
```

In [ ]:
from google.colab import files

files.download("app.py")
files.download("airline_model.pkl")